# Подготовка данных для разметки интентов

**Цель ноутбука.** Подготовить выборку русскоязычных диалогов из датасета `d0rj/dialogsum-ru` для последующей ручной разметки интентов в рамках магистерской диссертации «Семантический анализ русскоязычных диалогов для задачи распознавания намерений с улучшением на базе предобученных моделей».

**Что делает ноутбук:**

- подключает Google Drive в среде Google Colab;
- загружает датасет `d0rj/dialogsum-ru` из Hugging Face;
- отбирает воспроизводимую случайную выборку из 200 диалогов сплита `train`;
- считает простые метаданные по каждому диалогу (количество реплик, слов);
- добавляет пустые столбцы для ручной разметки интентов;
- сохраняет результирующий CSV на Google Drive по пути `data/annotation/dialogue_intent_annotation_v1.csv`.

Ноутбук **не выполняет** автоматическую классификацию интентов — интенты размечаются вручную.

Ноутбук рассчитан на запуск в Google Colab.

## 1. Подключение Google Drive и пути к артефактам

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/russian-dialogue-intent-thesis"
DATA_DIR = f"{BASE_DIR}/data"
ANNO_DIR = f"{DATA_DIR}/annotation"

os.makedirs(ANNO_DIR, exist_ok=True)
print(f"OK: {ANNO_DIR}")

## 2. Установка зависимостей и импорт библиотек

In [ ]:
# При необходимости устанавливаем библиотеку datasets
try:
    import datasets  # noqa: F401
except ImportError:
    !pip install -q datasets

In [ ]:
import os
import random

import numpy as np
import pandas as pd
from datasets import load_dataset

## 3. Загрузка датасета `d0rj/dialogsum-ru`

In [ ]:
# Загружаем train-сплит датасета
dataset = load_dataset("d0rj/dialogsum-ru")
train_ds = dataset["train"]

print(f"Размер train-сплита: {len(train_ds)}")
print(f"Колонки: {train_ds.column_names}")

## 4. Подготовка случайной выборки из 200 диалогов

In [ ]:
# Воспроизводимая случайная выборка
SEED = 42
SAMPLE_SIZE = 200

random.seed(SEED)
np.random.seed(SEED)

n_total = len(train_ds)
n_sample = min(SAMPLE_SIZE, n_total)

sample_indices = sorted(random.sample(range(n_total), n_sample))
print(f"Размер выборки: {n_sample} из {n_total}")

In [ ]:
# Вспомогательные функции, устойчивые к неожиданным форматам данных

def safe_get(row, key, default=""):
    """Безопасно достаём поле из строки, возвращаем default при отсутствии."""
    if row is None:
        return default
    try:
        value = row.get(key, default) if hasattr(row, "get") else row[key]
    except (KeyError, IndexError, TypeError):
        return default
    if value is None:
        return default
    return value


def count_utterances(dialogue_text):
    """Грубо считаем число реплик по строкам / маркерам говорящих."""
    if not isinstance(dialogue_text, str) or not dialogue_text.strip():
        return 0
    # Делим по переносам строк, отбрасываем пустые
    lines = [ln.strip() for ln in dialogue_text.splitlines() if ln.strip()]
    if len(lines) > 1:
        return len(lines)
    # Если переносов нет, пробуем найти маркеры вида '#Person1#:' или 'A:'
    import re
    markers = re.findall(r"#?[A-Za-zА-Яа-я0-9_]+#?\s*:", dialogue_text)
    if markers:
        return len(markers)
    return 1


def count_words(text):
    """Приблизительное число слов в тексте."""
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(text.split())

In [ ]:
# Формируем записи выборки
records = []
columns = set(train_ds.column_names)

for idx in sample_indices:
    row = train_ds[idx]

    if "id" in columns:
        dialog_id = safe_get(row, "id", default=str(idx))
    else:
        dialog_id = str(idx)

    dialogue = safe_get(row, "dialogue", default="") if "dialogue" in columns else ""
    summary = safe_get(row, "summary", default="") if "summary" in columns else ""

    if not isinstance(dialogue, str):
        dialogue = str(dialogue) if dialogue is not None else ""
    if not isinstance(summary, str):
        summary = str(summary) if summary is not None else ""

    records.append({
        "dialog_id": dialog_id,
        "split": "train",
        "dialogue": dialogue,
        "summary": summary,
        "n_utterances": count_utterances(dialogue),
        "dialogue_words": count_words(dialogue),
        "summary_words": count_words(summary) if summary else 0,
    })

print(f"Подготовлено записей: {len(records)}")

## 5. Добавление столбцов для ручной разметки

Важно: интенты **не** проставляются автоматически — все поля разметки остаются пустыми и заполняются вручную.

In [ ]:
df = pd.DataFrame(records)

# Пустые столбцы для ручной разметки
df["primary_intent"] = ""
df["secondary_intent"] = ""
df["is_ambiguous"] = 0
df["annotation_notes"] = ""
df["annotator"] = ""
df["label_version"] = "v1"

# Логический порядок колонок
column_order = [
    "dialog_id",
    "split",
    "dialogue",
    "summary",
    "n_utterances",
    "dialogue_words",
    "summary_words",
    "primary_intent",
    "secondary_intent",
    "is_ambiguous",
    "annotation_notes",
    "annotator",
    "label_version",
]
df = df[column_order]

print(f"Форма таблицы: {df.shape}")
df.head(3)

## 6. Сохранение CSV на Google Drive

In [ ]:
csv_path = f"{ANNO_DIR}/dialogue_intent_annotation_v1.csv"
df.to_csv(csv_path, sep=",", encoding="utf-8", index=False)
print(f"Сохранено: {csv_path}")

## 7. Проверка сохранённого CSV

In [ ]:
df_check = pd.read_csv(csv_path, sep=",", encoding="utf-8")
print(f"Форма: {df_check.shape}")
print(f"Колонки: {list(df_check.columns)}")
df_check.head(5)